In [30]:
import os, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = torch.device('xpu' if torch.xpu.is_available() else 'cpu')

D_MODEL, NHEAD, NUM_LAYERS, FFN_DIM, DROPOUT = 128, 4, 2, 256, 0.1
BATCH_SIZE, MAX_EPOCHS, PATIENCE, LR = 256, 50, 5, 1e-3
MAX_SEQ_LEN, MASK_PROB = 20, 0.15
TSNE_BG, TSNE_N_ADDR = 2000, 5

SAVE = True
PATH_SAVE_MODEL = '../data/02_artifacts/model.pt'
PATH_LOAD_MODEL = '../data/02_artifacts/model.pt'  # set to a path string to skip training and load saved weights
print(f'device: {device}')

device: xpu


In [31]:
RAW_PATH = '../data/01_raw/311_90days.parquet'
os.makedirs('../data/01_raw', exist_ok=True)

if os.path.exists(RAW_PATH):
    df = pd.read_parquet(RAW_PATH)
    print(f'Loaded {len(df):,} records from cache')
else:
    import requests_cache
    from datetime import datetime, timedelta, timezone
    session = requests_cache.CachedSession('../data/00_cache/311', expire_after=3600)
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=90)
    where = (
        f"created_date >= '{start.strftime('%Y-%m-%dT00:00:00')}'"
        f" and created_date <= '{end.strftime('%Y-%m-%dT23:59:59')}'"
    )
    url = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'
    frames = []
    for offset in range(0, 2_000_000, 50000):
        r = session.get(url, params={'$where': where, '$order': 'created_date ASC',
                                     '$limit': '50000', '$offset': str(offset)}, timeout=120)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        frames.append(pd.DataFrame(batch))
        print(f'  fetched {offset + len(batch):,}')
        if len(batch) < 50000:
            break
    df = pd.concat(frames, ignore_index=True)
    df['created_date'] = pd.to_datetime(df['created_date'])
    for col in ['latitude', 'longitude']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'location' in df.columns:
        df = df.drop(columns=['location'])
    df.to_parquet(RAW_PATH)
    print(f'Fetched and saved {len(df):,} records')

Loaded 100,000 records from cache


In [32]:
df_clean = df.dropna(subset=['incident_address', 'complaint_type', 'descriptor']).copy()
df_clean['created_date'] = pd.to_datetime(df_clean['created_date'])

closed_flag = df_clean['closed_date'].notna().map({True: 'CLOSED', False: 'OPEN'}) \
    if 'closed_date' in df_clean.columns else 'OPEN'
df_clean['token'] = (
    df_clean['complaint_type'].str.upper() + '_' +
    df_clean['descriptor'].str.upper() + '_' +
    closed_flag
)
df_clean['year_month'] = df_clean['created_date'].dt.to_period('M')

all_months = pd.period_range(
    df_clean['created_date'].min().to_period('M'),
    df_clean['created_date'].max().to_period('M'),
    freq='M',
)
month_strs = [str(m) for m in all_months]
print(f'Months: {month_strs}')

# Group by address+month, sort by created_date, cap at MAX_SEQ_LEN
monthly = (
    df_clean.sort_values('created_date')
    .groupby(['incident_address', 'year_month'])['token']
    .apply(lambda x: list(x)[:MAX_SEQ_LEN])
    .reset_index()
)
monthly.columns = ['address', 'month', 'tokens']
monthly['month'] = monthly['month'].astype(str)

# Full (address x month) grid; fill missing with NO_CALLS
addresses = df_clean['incident_address'].unique()
full_idx = pd.MultiIndex.from_product([addresses, month_strs], names=['address', 'month'])
monthly_full = (
    pd.DataFrame(index=full_idx).reset_index()
    .merge(monthly, on=['address', 'month'], how='left')
)
monthly_full['tokens'] = monthly_full['tokens'].apply(
    lambda x: x if isinstance(x, list) else ['NO_CALLS']
)
print(f'{len(monthly_full):,} (address, month) samples  |  {len(addresses):,} unique addresses')

Months: ['2026-03', '2026-04']
95,632 (address, month) samples  |  47,816 unique addresses


In [33]:
unique_addrs = monthly_full['address'].unique().tolist()
train_a, temp_a = train_test_split(unique_addrs, test_size=0.2, random_state=42)
val_a,  test_a  = train_test_split(temp_a,       test_size=0.5, random_state=42)

split_map = {**{a: 'train' for a in train_a}, **{a: 'val' for a in val_a}, **{a: 'test' for a in test_a}}
monthly_full['split'] = monthly_full['address'].map(split_map)

train_df = monthly_full[monthly_full['split'] == 'train']
val_df   = monthly_full[monthly_full['split'] == 'val']
test_df  = monthly_full[monthly_full['split'] == 'test']
print(f'train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')

token_freq = Counter(t for tokens in train_df['tokens'] for t in tokens)
SPECIAL = ['[PAD]', '[MASK]', '[UNK]', 'NO_CALLS']
vocab = {t: i for i, t in enumerate(SPECIAL)}
for tok in token_freq:
    if tok not in vocab:
        vocab[tok] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}
print(f'Vocab size: {len(vocab)}')

train=76,504  val=9,564  test=9,564
Vocab size: 855


In [34]:
class MLMDataset(Dataset):
    def __init__(self, records, vocab, max_len=MAX_SEQ_LEN, mask_prob=MASK_PROB):
        self.records = records
        self.vocab = vocab
        self.max_len = max_len
        self.mask_prob = mask_prob
        self.pad_id  = vocab['[PAD]']
        self.mask_id = vocab['[MASK]']
        self.unk_id  = vocab['[UNK]']

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        toks = rec['tokens'][:self.max_len]
        ids  = [self.vocab.get(t, self.unk_id) for t in toks]
        seq_len = len(ids)
        attn = [1] * seq_len + [0] * (self.max_len - seq_len)
        ids  = ids + [self.pad_id] * (self.max_len - seq_len)
        labels, inp = [-100] * self.max_len, ids[:]
        for i in range(seq_len):
            if random.random() < self.mask_prob:
                labels[i] = inp[i]
                inp[i] = self.mask_id
        return {
            'input_ids':      torch.tensor(inp,    dtype=torch.long),
            'attention_mask': torch.tensor(attn,   dtype=torch.long),
            'labels':         torch.tensor(labels, dtype=torch.long),
            'address': rec['address'],
            'month':   rec['month'],
        }


class AddressTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=NUM_LAYERS, ffn_dim=FFN_DIM, dropout=DROPOUT):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ffn_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids)
        x = self.encoder(x, src_key_padding_mask=(attention_mask == 0))
        return self.head(x), x

    def encode(self, input_ids, attention_mask):
        with torch.no_grad():
            x = self.embed(input_ids)
            x = self.encoder(x, src_key_padding_mask=(attention_mask == 0))
            m = attention_mask.unsqueeze(-1).float()
            return (x * m).sum(1) / m.sum(1).clamp(min=1)

In [35]:
def collate_fn(batch):
    out = {}
    for k in batch[0]:
        out[k] = torch.stack([b[k] for b in batch]) if isinstance(batch[0][k], torch.Tensor) \
                 else [b[k] for b in batch]
    return out

train_ds = MLMDataset(train_df.to_dict('records'), vocab)
val_ds   = MLMDataset(val_df.to_dict('records'),   vocab)
test_ds  = MLMDataset(test_df.to_dict('records'),  vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

model = AddressTransformer(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}')


def compute_ppl(model, loader):
    model.eval()
    t_loss = t_tok = 0
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            logits, _ = model(ids, mask)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), lbls.view(-1),
                                   ignore_index=-100, reduction='sum')
            t_loss += loss.item()
            t_tok  += (lbls != -100).sum().item()
    return math.exp(t_loss / max(t_tok, 1))

Model params: 484,695
Train batches: 299  Val batches: 38


In [36]:
if PATH_LOAD_MODEL is not None:
    model.load_state_dict(torch.load(PATH_LOAD_MODEL, map_location=device))
    print(f'Loaded model from {PATH_LOAD_MODEL}')
else:
    best_ppl, best_state, no_improve = float('inf'), None, 0
    history = []

    for epoch in range(MAX_EPOCHS):
        model.train()
        t_loss = t_tok = 0
        for batch in train_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            logits, _ = model(ids, mask)
            n_tok = (lbls != -100).sum().item()
            loss  = F.cross_entropy(logits.view(-1, logits.size(-1)), lbls.view(-1), ignore_index=-100)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            t_loss += loss.item() * n_tok
            t_tok  += n_tok

        train_ppl = math.exp(t_loss / max(t_tok, 1))
        val_ppl   = compute_ppl(model, val_loader)
        history.append((train_ppl, val_ppl))
        print(f'epoch {epoch+1:3d}  train_ppl={train_ppl:.2f}  val_ppl={val_ppl:.2f}')

        if val_ppl < best_ppl:
            best_ppl   = val_ppl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  early stop at epoch {epoch+1}  best_val_ppl={best_ppl:.2f}')
                break

    model.load_state_dict(best_state)

    if SAVE:
        os.makedirs(os.path.dirname(PATH_SAVE_MODEL), exist_ok=True)
        torch.save(model.state_dict(), PATH_SAVE_MODEL)
        print(f'Saved model to {PATH_SAVE_MODEL}')

    train_ppls, val_ppls = zip(*history)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_ppls, label='train')
    ax.plot(val_ppls,   label='val')
    ax.set_xlabel('epoch'); ax.set_ylabel('perplexity'); ax.set_title('Training curve')
    ax.legend(); plt.tight_layout(); plt.show()

Loaded model from ../data/02_artifacts/model.pt


In [37]:
model.eval()
t_loss = t_correct = t_tok = 0
with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        logits, _ = model(ids, mask)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), lbls.view(-1),
                               ignore_index=-100, reduction='sum')
        preds = logits.argmax(-1)
        valid = lbls != -100
        t_correct += (preds[valid] == lbls[valid]).sum().item()
        t_loss    += loss.item()
        t_tok     += valid.sum().item()

test_ppl = math.exp(t_loss / max(t_tok, 1))
test_acc = t_correct / max(t_tok, 1)
print(f'Test perplexity:         {test_ppl:.4f}')
print(f'Masked token accuracy:   {test_acc:.4f}')

Test perplexity:         20.8309
Masked token accuracy:   0.4795


In [38]:
model.eval()
all_embs, all_meta = [], []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        embs = model.encode(ids, mask).cpu().numpy()
        all_embs.append(embs)
        for i in range(len(batch['address'])):
            all_meta.append({'address': batch['address'][i], 'month': batch['month'][i]})

all_embs = np.vstack(all_embs)
emb_df = pd.DataFrame(all_meta)

# Tag each embedding as NO_CALLS or active
addr_month_tokens = monthly_full.set_index(['address', 'month'])['tokens']
emb_df['no_calls'] = [
    addr_month_tokens.get((r['address'], r['month']), ['NO_CALLS']) == ['NO_CALLS']
    for _, r in emb_df.iterrows()
]
print(f'Embeddings: {all_embs.shape}  no_calls={emb_df["no_calls"].sum():,}')

Embeddings: (9564, 128)  no_calls=4,182


In [39]:
target_month = month_strs[-1]
month_mask = (emb_df['month'] == target_month).values
month_embs = all_embs[month_mask]
month_meta = emb_df[month_mask].reset_index(drop=True)

rng_idx = np.random.choice(len(month_embs), size=min(TSNE_BG + 500, len(month_embs)), replace=False)
coords = TSNE(n_components=2, random_state=42, perplexity=min(30, len(rng_idx)-1)).fit_transform(month_embs[rng_idx])

sub_meta   = month_meta.iloc[rng_idx]
status     = np.where(sub_meta['no_calls'].values, 'NO_CALLS', 'active')
tok_lookup = monthly_full.set_index(['address', 'month'])['tokens']

hover = []
for a in sub_meta['address']:
    toks = tok_lookup.get((a, target_month), ['NO_CALLS'])
    hover.append(f'<b>{a}</b><br>month: {target_month}<br>' + '<br>'.join(toks))

fig = go.Figure()
for s, color in [('active', '#2563eb'), ('NO_CALLS', '#aaaaaa')]:
    sel = status == s
    fig.add_trace(go.Scatter(
        x=coords[sel, 0], y=coords[sel, 1],
        mode='markers',
        name=s,
        marker=dict(color=color, size=5, opacity=0.5),
        text=np.array(hover)[sel],
        hoverinfo='text',
    ))

fig.update_layout(
    title=f'TSNE – test embeddings for {target_month}  (n={len(rng_idx):,})',
    xaxis_title='TSNE-1',
    yaxis_title='TSNE-2',
    width=900, height=700,
)
fig.show()

In [41]:
from umap import UMAP as UMAP_

N_MIN_CALLS = 5
UMAP_THRESH = 2.0

# Filter 1: active in ALL months
active_per_addr = (
    monthly_full[(monthly_full['split'] == 'test') &
                 monthly_full['tokens'].apply(lambda x: x != ['NO_CALLS'])]
    .groupby('address').size()
)
consistent_addrs = set(active_per_addr[active_per_addr == len(month_strs)].index)

# Filter 2: at least one month with >= N_MIN_CALLS calls
max_calls = (
    monthly_full[monthly_full['split'] == 'test']
    .assign(n=lambda df: df['tokens'].apply(len))
    .groupby('address')['n'].max()
)
high_activity = set(max_calls[max_calls >= N_MIN_CALLS].index)

candidates = list(consistent_addrs & high_activity)
print(f'Candidates after activity filters: {len(candidates)}')

# Filter 3: UMAP — keep addresses whose month embeddings move >= UMAP_THRESH apart
umap_coords = UMAP_(n_components=2, random_state=42).fit_transform(all_embs)
umap_df = emb_df.copy()
umap_df['ux'] = umap_coords[:, 0]
umap_df['uy'] = umap_coords[:, 1]

def max_umap_jump(addr):
    pts = umap_df[umap_df['address'] == addr].sort_values('month')[['ux', 'uy']].values
    return np.linalg.norm(np.diff(pts, axis=0), axis=1).max() if len(pts) >= 2 else 0.0

candidates = [a for a in candidates if max_umap_jump(a) >= UMAP_THRESH]
sample_addrs = random.sample(candidates, min(15, len(candidates)))
print(f'Candidates after UMAP filter: {len(candidates)}')

fg_mask = emb_df['address'].isin(sample_addrs).values
bg_pool = np.where(~fg_mask)[0]
bg_idx  = np.random.choice(bg_pool, size=min(TSNE_BG, len(bg_pool)), replace=False)
fg_idx  = np.where(fg_mask)[0]
tsne_idx = np.concatenate([bg_idx, fg_idx])

coords_all = TSNE(n_components=2, random_state=42,
                  perplexity=min(30, len(tsne_idx)-1)).fit_transform(all_embs[tsne_idx])
n_bg = len(bg_idx)

tsne_frame = emb_df.iloc[tsne_idx].copy().reset_index(drop=True)
tsne_frame['x'] = coords_all[:, 0]
tsne_frame['y'] = coords_all[:, 1]

tok_lookup    = monthly_full.set_index(['address', 'month'])['tokens']
cmap          = plt.cm.tab10.colors
month_symbols = ['circle', 'square', 'diamond', 'cross', 'x']

fig = go.Figure()

bg = tsne_frame.iloc[:n_bg]
fig.add_trace(go.Scatter(
    x=bg['x'], y=bg['y'],
    mode='markers',
    name='background',
    marker=dict(color='#e5e7eb', size=4, opacity=0.4),
    hoverinfo='skip',
    showlegend=False,
))

for ai, addr in enumerate(sample_addrs):
    addr_rows = tsne_frame[tsne_frame['address'] == addr].sort_values('month')
    r, g, b = [int(c * 255) for c in cmap[ai % len(cmap)][:3]]
    color = f'rgb({r},{g},{b})'

    hover = []
    for _, row in addr_rows.iterrows():
        toks = tok_lookup.get((row['address'], row['month']), ['NO_CALLS'])
        hover.append(f'<b>{addr}</b><br>month: {row["month"]}<br>' + '<br>'.join(toks))

    symbols = [month_symbols[mi % len(month_symbols)] for mi in range(len(addr_rows))]

    fig.add_trace(go.Scatter(
        x=addr_rows['x'].values,
        y=addr_rows['y'].values,
        mode='lines+markers',
        name=f'addr {ai+1}',
        line=dict(color=color, width=2),
        marker=dict(color=color, size=12, symbol=symbols),
        text=hover,
        hoverinfo='text',
    ))

    for i in range(len(addr_rows) - 1):
        fig.add_annotation(
            x=addr_rows.iloc[i+1]['x'], y=addr_rows.iloc[i+1]['y'],
            ax=addr_rows.iloc[i]['x'],  ay=addr_rows.iloc[i]['y'],
            xref='x', yref='y', axref='x', ayref='y',
            arrowhead=2, arrowsize=1.2, arrowwidth=2,
            arrowcolor=color, showarrow=True,
        )

fig.update_layout(
    title='3-month embedding trajectories (consistently active, high-movement addresses)',
    xaxis_title='TSNE-1',
    yaxis_title='TSNE-2',
    width=1000, height=800,
    legend=dict(font=dict(size=9)),
)
fig.show()

Candidates after activity filters: 129


/home/zaccosenza/code/project-311/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/zaccosenza/code/project-311/.venv/lib/python3.12/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


Candidates after UMAP filter: 65
